In [279]:
import pandas as pd

In [280]:
df = pd.read_parquet("../data/processed/dvf_77_2025.parquet")
print(df.shape)
df.dtypes

(64913, 40)


id_mutation                         str
date_mutation                       str
numero_disposition                int64
nature_mutation                     str
valeur_fonciere                 float64
adresse_numero                  float64
adresse_suffixe                     str
adresse_nom_voie                    str
adresse_code_voie                   str
code_postal                         str
code_commune                        str
nom_commune                         str
code_departement                    str
ancien_code_commune                 str
ancien_nom_commune                  str
id_parcelle                         str
ancien_id_parcelle                  str
numero_volume                       str
lot1_numero                         str
lot1_surface_carrez             float64
lot2_numero                         str
lot2_surface_carrez             float64
lot3_numero                         str
lot3_surface_carrez             float64
lot4_numero                         str


In [281]:
valeurs_uniques = df['nature_mutation'].unique()
print(valeurs_uniques)

<ArrowStringArray>
[                             'Vente', 'Vente en l'état futur d'achèvement',
              'Vente terrain à bâtir',                            'Echange',
                       'Adjudication',                      'Expropriation']
Length: 6, dtype: str


In [282]:
nbr_vente = df['nature_mutation'].value_counts()
print(nbr_vente)

nature_mutation
Vente                                 58727
Vente en l'état futur d'achèvement     5117
Echange                                 693
Vente terrain à bâtir                   187
Adjudication                            175
Expropriation                            14
Name: count, dtype: int64


In [283]:
# Groupement par nature de mutation et comptage des id_mutation uniques
resultat = df.groupby('nature_mutation')['id_mutation'].nunique()

print(resultat)

nature_mutation
Adjudication                             62
Echange                                 407
Expropriation                            14
Vente                                 21317
Vente en l'état futur d'achèvement     1677
Vente terrain à bâtir                    53
Name: id_mutation, dtype: int64


In [284]:
df_vente = df[df['nature_mutation'] == 'Vente']
print(df_vente.shape)

(58727, 40)


In [285]:
# Groupement par nature de mutation et comptage des id_mutation uniques
resultat = df_vente.groupby('nature_mutation')['id_mutation'].nunique()

print(resultat)

nature_mutation
Vente    21317
Name: id_mutation, dtype: int64


In [286]:
print(df.groupby("id_mutation")["nature_mutation"].nunique().max())

2


In [287]:
natures_par_vente = df.groupby("id_mutation")["nature_mutation"].nunique()
natures_par_vente[natures_par_vente > 1]

id_mutation
2025-1050579    2
Name: nature_mutation, dtype: int64

In [288]:
colonnes = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
            "nom_commune", "type_local", "surface_reelle_bati", "nature_culture"]
df[df["id_mutation"] == "2025-1050579"][colonnes]

,id_mutation,date_mutation,nature_mutation,valeur_fonciere,nom_commune,type_local,surface_reelle_bati,nature_culture
25823,2025-1050579,2025-12-23,Echange,1.0,Pontault Combault,NaN,NaN,sols
25824,2025-1050579,2025-12-23,Echange,1.0,Pontault Combault,NaN,NaN,sols
25825,2025-1050579,2025-12-23,Vente,1.0,Melun,NaN,NaN,NaN
25826,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25827,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25828,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25829,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25830,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25831,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25832,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN


In [289]:
# 1. Identifiants des ventes qui portent plusieurs natures
ids_a_exclure = natures_par_vente[natures_par_vente > 1].index

# 2. DataFrame sans ces ventes
df_coherent = df[~df["id_mutation"].isin(ids_a_exclure)]

# 3. Vérifications
print(df_coherent.groupby("id_mutation")["nature_mutation"].nunique().max())
print(len(df) - len(df_coherent))



1
13


In [290]:
df_vente = df_coherent[df_coherent['nature_mutation'] == 'Vente'].copy()


In [291]:
print(df_vente.shape)

(58716, 40)


In [292]:
resultat = df_vente.groupby('nature_mutation')['id_mutation'].nunique()
print(resultat)

nature_mutation
Vente    21316
Name: id_mutation, dtype: int64


In [293]:
df_vente["est_logement"] = df_vente["type_local"].isin(["Maison", "Appartement"])
logements_par_vente = df_vente.groupby("id_mutation")["est_logement"].sum()
logements_par_vente.head()

id_mutation
2025-1040912    1
2025-1040913    1
2025-1040914    1
2025-1040915    1
2025-1040916    1
Name: est_logement, dtype: int64

In [294]:
logements_par_vente.value_counts().sort_index()

est_logement
0       4892
1      13950
2       2163
3        135
4         84
5         17
6         31
7         11
8          4
9          6
10         1
11         1
12         3
14         1
16         1
18         1
19         1
28         1
36         1
43         1
44         1
46         1
56         1
65         1
73         1
76         1
88         1
97         1
137        1
146        1
258        1
Name: count, dtype: int64

In [295]:
logement_258  = logements_par_vente[logements_par_vente == 258]
vente_258 = df_vente[df_vente["id_mutation"].isin(logement_258.index)]

In [296]:
print(vente_258["valeur_fonciere"].unique())
print(vente_258["nom_commune"].unique())
print(vente_258["type_local"].value_counts(dropna=False))

[579123.36]
<ArrowStringArray>
['Noisiel']
Length: 1, dtype: str
type_local
Appartement                                 258
Dépendance                                  102
Local industriel. commercial ou assimilé      2
NaN                                           1
Name: count, dtype: int64


In [297]:
logement_1 = logements_par_vente[logements_par_vente == 1]

vente_1 = df_vente[df_vente["id_mutation"].isin(logement_1.index)]

In [298]:
print(vente_1["type_local"].value_counts(dropna=False))

print(vente_1.head())

type_local
Dépendance                                  11810
Maison                                       7881
Appartement                                  6069
NaN                                          5086
Local industriel. commercial ou assimilé      268
Name: count, dtype: int64
    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2025-1040912    2025-01-06                   1           Vente   
1  2025-1040912    2025-01-06                   1           Vente   
2  2025-1040912    2025-01-06                   1           Vente   
3  2025-1040913    2025-01-03                   1           Vente   
4  2025-1040913    2025-01-03                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe     adresse_nom_voie  \
0         121000.0            31.0             NaN       RUE SERMONOISE   
1         121000.0            31.0             NaN       RUE SERMONOISE   
2         121000.0            31.0             NaN       RUE SERMONOISE  

In [299]:
vente_1["a_local"] = vente_1["type_local"].isin(["Local industriel. commercial ou assimilé"])
vente_local = vente_1.groupby("id_mutation")["a_local"].any().sum()
print(vente_local)

199


In [300]:
vente_local = vente_1.groupby("id_mutation")["a_local"].any()
ids_avec_local = vente_local[vente_local].index

df_regle1 = vente_1[~vente_1["id_mutation"].isin(ids_avec_local)].copy()
print(df_regle1["id_mutation"].nunique())

13751


In [301]:
verif_logement = df_regle1["type_local"].value_counts(dropna= False)
print(verif_logement)

type_local
Dépendance     11636
Maison          7751
Appartement     6000
NaN             4803
Name: count, dtype: int64


In [302]:
df_regle1["est_logement"] = df_regle1["type_local"].isin(["Maison","Appartement"])

verif_1_logement = df_regle1.groupby("id_mutation")["est_logement"].sum()

print(verif_1_logement.max())
print(verif_1_logement.min())

1
1


In [303]:
df_logements = df_regle1[df_regle1["type_local"].isin(["Maison","Appartement"])].copy()
print(df_logements["id_mutation"].nunique() == len(df_logements))
print(df_logements["valeur_fonciere"].isna().sum())
print(df_logements["surface_reelle_bati"].isna().sum())
print((df_logements["surface_reelle_bati"] == 0.0).sum())

True
28
1
0


In [304]:
sans_prix = df_logements[df_logements["valeur_fonciere"].isna()]
sans_surface = df_logements[df_logements["surface_reelle_bati"].isna()] 

ids_sans_prix_ou_surface = pd.concat([sans_surface["id_mutation"] , sans_prix["id_mutation"]])

df_regle123 =  df_regle1[~df_regle1["id_mutation"].isin(ids_sans_prix_ou_surface)].copy()

df_logements3 = df_regle123[df_regle123["type_local"].isin(["Maison","Appartement"])].copy()
print(df_logements3["id_mutation"].nunique() == len(df_logements3))
print(df_logements3["valeur_fonciere"].isna().sum())
print(df_logements3["surface_reelle_bati"].isna().sum())
print((df_logements3["surface_reelle_bati"] == 0.0).sum())
print(len(df_logements3))

True
0
0
0
13722


In [305]:
df_logements3["prix_m2"] = df_logements3["valeur_fonciere"] / df_logements3["surface_reelle_bati"] 
print(df_logements3["prix_m2"].describe())


count     13722.000000
mean       3146.983883
std        2797.664085
min           0.012500
25%        2380.952381
50%        3069.760770
75%        3767.248062
max      300000.000000
Name: prix_m2, dtype: float64


In [306]:
df_regle123["est_dependances"] = df_regle123["type_local"].isin(["Dépendance"])

nb_dependances = df_regle123.groupby("id_mutation")["est_dependances"].sum()

nb_dependances = nb_dependances.rename("nb_dependances").reset_index()

df_final = pd.merge(df_logements3, nb_dependances, on="id_mutation", how="left")

print(len(df_final))

print(df_final["nb_dependances"].value_counts().sort_index())

print(df_final["nb_dependances"].isna().sum())

13722
nb_dependances
0     5734
1     5050
2     2383
3      458
4       79
5       10
6        3
7        2
10       1
18       1
19       1
Name: count, dtype: int64
0


In [307]:
colonnes = ["id_mutation", "type_local", "surface_reelle_bati", "id_parcelle", "nature_culture", "surface_terrain"]

# 1. Dans df_final (une ligne par vente) : les maisons avec dépendance(s) et terrain
conditions = (df_final["nb_dependances"] > 0) & (df_final["type_local"] == "Maison") & (df_final["surface_terrain"] > 0)
ids_exemples = df_final.loc[conditions, "id_mutation"]

# 2. On prend la première de ces ventes
exemple = ids_exemples.iloc[0]

# 3. On affiche TOUTES ses lignes, dans df_regle123 (qui contient aussi dépendances et terrains)
df_regle123.loc[df_regle123["id_mutation"] == exemple, colonnes]

,id_mutation,type_local,surface_reelle_bati,id_parcelle,nature_culture,surface_terrain
23,2025-1040923,Maison,83.0,77296000AC0478,sols,104.0
24,2025-1040923,Dépendance,0.0,77296000AC0478,sols,104.0
25,2025-1040923,NaN,NaN,77296000AC0479,sols,17.0
26,2025-1040923,NaN,NaN,77296000AC0832,sols,68.0


In [ ]:
terrains = df_regle123[["id_mutation", "id_parcelle", "nature_culture", "surface_terrain"]].drop_duplicates()

terrain_par_vente = terrains.groupby("id_mutation")["surface_terrain"].sum()

terrain_par_vente = terrain_par_vente.rename("surface_terrain_total").reset_index()


df_final = pd.merge(df_final, terrain_par_vente, on="id_mutation", how="left")

print(len(df_final))

colonnes2 = ["id_mutation", "type_local", "surface_reelle_bati", "id_parcelle", "nature_culture", "surface_terrain", "surface_terrain_total"]

conditions2 = (df_final["nb_dependances"] > 0) & (df_final["type_local"] == "Maison") & (df_final["surface_terrain"] > 0)


resultat = df_final.loc[conditions2, colonnes2]

print(resultat.head())

13722
     id_mutation type_local  surface_reelle_bati     id_parcelle  \
11  2025-1040923     Maison                 83.0  77296000AC0478   
16  2025-1040935     Maison                 58.0  77495000BE0080   
18  2025-1040938     Maison                 89.0  77053000AR0358   
19  2025-1040940     Maison                127.0  77501000AD0016   
20  2025-1040943     Maison                152.0  77458000BH0360   

   nature_culture  surface_terrain  surface_terrain_total  
11           sols            104.0                  189.0  
16           sols            394.0                  394.0  
18           sols            243.0                  243.0  
19           sols            549.0                 2181.0  
20           sols            200.0                  547.0  


In [309]:
apparts = df_final[df_final["type_local"] == "Appartement"]
print(apparts["surface_terrain_total"].isna().sum())
print((apparts["surface_terrain_total"] == 0).sum())
print(len(apparts))

0
5760
5991


In [310]:
print(df_final.columns)

Index(['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation',
       'valeur_fonciere', 'adresse_numero', 'adresse_suffixe',
       'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune',
       'nom_commune', 'code_departement', 'ancien_code_commune',
       'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle',
       'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero',
       'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez',
       'lot4_numero', 'lot4_surface_carrez', 'lot5_numero',
       'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local',
       'surface_reelle_bati', 'nombre_pieces_principales',
       'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale',
       'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude',
       'est_logement', 'a_local', 'prix_m2', 'nb_dependances',
       'surface_terrain_total'],
      dtype='str')


In [312]:
colonnes_finales = [
    # Identification de la vente
    "id_mutation",
    "date_mutation",

    # Prix
    "valeur_fonciere",
    "prix_m2",

    # Caractéristiques du bien
    "type_local",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "nb_dependances",
    "surface_terrain_total",

    # Localisation
    "code_commune",
    "nom_commune",
    "code_postal",
    "longitude",
    "latitude",
]

df_propre = df_final[colonnes_finales]

df_propre.to_parquet("../data/processed/dvf_77_2025_propre.parquet", index=False)

In [313]:
df_verif = pd.read_parquet("../data/processed/dvf_77_2025_propre.parquet")
print(df_verif.shape)
df_verif.dtypes

(13722, 14)


id_mutation                      str
date_mutation                    str
valeur_fonciere              float64
prix_m2                      float64
type_local                       str
surface_reelle_bati          float64
nombre_pieces_principales    float64
nb_dependances                 int64
surface_terrain_total        float64
code_commune                     str
nom_commune                      str
code_postal                      str
longitude                    float64
latitude                     float64
dtype: object